[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](
https://colab.research.google.com/github/Simone-Alghisi/HMD-Lab/blob/master/notebooks/2_nlu.ipynb)

On Colab:
1. Switch to a GPU Runtime by clicking on *Runtime > Change runtime type > T4 GPU*
2. Run the cell below

In [ ]:
# For Google Colab only
!git clone https://github.com/Simone-Alghisi/HMD-Lab.git
%cd /content/HMD-Lab/notebooks 

![system_architecture](./assets/system.jpg)

# Dialogue Manager (DM)

The Dialogue Manager (DM) decides the next best action given the current [Dialogue State](./2_nlu.ipynb) (or NLU output). It sits between the NLU and the NLG components and is responsible for 
- taking the "best" decision and passing it to the other components
- handling errors (e.g., OOD requests or failing components) and ambiguities (e.g., multiple values for a single slot)

Typical DM responsibilities:
- Decide whether to ask for missing slots (e.g., `pizza_type`)
- Provide information or options (e.g., the menu)
- Solve ambiguities or errors (e.g., wrong or invalid values for a given slot)
- Confirm a completed intent
- Trigger external services (e.g., contact an external database/API)

## Question

You want to design an OrderBot to collect the user's pizza order.

Consider the following dialogue state:

```json
{
  "intent": "pizza_ordering",
  "slots": {
    "pizza_size": "medium",
    "pizza_type": "margherita",
    "pizza_count": null
  }
}
```

1. What is the next best action?
2. What if more than one slot is missing?
3. What happens if the intent becomes `out_of_domain`?


### Solution

1. Ask for `pizza_count` since it's the only slot missing to complete the order
2. Ask one at a time to not be over-informative
3. Proceed with fallback policy (e.g., tell the NLG to explain the system capabilities to the user)

## Code

In [1]:
import sys

sys.path.append("..")

from utils import MODELS
from transformers import AutoTokenizer

model_name, InitModel, prepare_text = MODELS["qwen3"]

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = InitModel(
    model_name,
    dtype="auto",
    device_map="cuda:0",
)

/home/simone/miniconda3/envs/hmd/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 3/3 [00:01<00:00,  1.96it/s]


We ask the DM to output a single, compact action chosen from a small set. Example actions (the DM should output exactly one):

- `request_info(slot)`: ask the user for a missing slot, e.g. `request_info(pizza_size)`
- `provide_info(intent, slot)`: provide possible values, e.g. `provide_info(pizza_ordering, pizza_size)`
- `confirmation(intent)`: confirm filled slots for an intent, e.g. `confirmation(pizza_ordering)`

In [2]:
import torch

from notebooks.notebook_utils import display_conversation
from models.qwen3 import prepare_text

task_prompt = """You are given the dialogue state for the current turn.
The dialogue state contains information about the user's intent and the extracted slot-value pairs:
{
    "intent": "...", 
    "slots": {
        "slot": "value"
    }
}

Based on the provided dialogue state, select the next best action from the list below:
- request_info(slot), if a slot value is missing (i.e., null)
- provide_info(intent, slot), provide the list of possible values for the requested slot for the given intent
- confirmation(intent), if all slots have been filled

Substitute 'slot' and 'intent' with the actual slot name and intent name from the dialogue state.
Only respond with the action in the exact format specified above. Do not include any additional text or explanation.
"""

In [3]:
ds = """{
  "intent": "pizza_ordering",
  "slots": {
    "pizza_size": null,
    "pizza_type": "margherita",
    "pizza_count": null
  }
}"""

messages = [
    {
        "role": "system", 
        "content": task_prompt
    }
]

text = prepare_text(ds, tokenizer, messages)

model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

with torch.no_grad():
    generated_ids = model.generate(**model_inputs).cpu()

# decode the output
output_ids = generated_ids[0][len(model_inputs.input_ids[0]) :].tolist()
content = tokenizer.decode(output_ids, skip_special_tokens=True)
display_conversation(messages, ds, content)


### Conversation

**System:** You are given the dialogue state for the current turn.
The dialogue state contains information about the user's intent and the extracted slot-value pairs:
{
    "intent": "...", 
    "slots": {
        "slot": "value"
    }
}

Based on the provided dialogue state, select the next best action from the list below:
- request_info(slot), if a slot value is missing (i.e., null)
- provide_info(intent, slot), provide the list of possible values for the requested slot for the given intent
- confirmation(intent), if all slots have been filled

Substitute 'slot' and 'intent' with the actual slot name and intent name from the dialogue state.
Only respond with the action in the exact format specified above. Do not include any additional text or explanation.


**User:** {
  "intent": "pizza_ordering",
  "slots": {
    "pizza_size": null,
    "pizza_type": "margherita",
    "pizza_count": null
  }
}

**Assistant:** request_info(pizza_size)

In [ ]:
ds = """{
  "intent": "request_info",
  "slots": "pizza_type"
}"""

messages = [
    {
        "role": "system", 
        "content": task_prompt
    }
]

text = prepare_text(ds, tokenizer, messages)

model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

with torch.no_grad():
    generated_ids = model.generate(**model_inputs).cpu()

# decode the output
output_ids = generated_ids[0][len(model_inputs.input_ids[0]) :].tolist()
content = tokenizer.decode(output_ids, skip_special_tokens=True)
display_conversation(messages, ds, content)


### Conversation

**System:** You are given the dialogue state for the current turn.
The dialogue state contains information about the user's intent and the extracted slot-value pairs:
{
    "intent": "...", 
    "slots": {
        "slot": "value"
    }
}

Based on the provided dialogue state, select the next best action from the list below:
- request_info(slot), if a slot value is missing (i.e., null)
- provide_info(intent, slot), provide the list of possible values for the requested slot for the given intent
- confirmation(intent), if all slots have been filled

Substitute 'slot' and 'intent' with the actual slot name and intent name from the dialogue state.
Only respond with the action in the exact format specified above. Do not include any additional text or explanation.


**User:** {
  "intent": "request_info",
  "slots": "pizza_type"
}

**Assistant:** request_info(pizza_type)

In [ ]:
ds = """{
  "intent": "pizza_ordering",
  "slots": {
    "pizza_size": "medium",
    "pizza_type": "margherita",
    "pizza_count": "1"
  }
}"""

messages = [
    {
        "role": "system", 
        "content": task_prompt
    }
]

text = prepare_text(ds, tokenizer, messages)

model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

with torch.no_grad():
    generated_ids = model.generate(**model_inputs).cpu()

# decode the output
output_ids = generated_ids[0][len(model_inputs.input_ids[0]) :].tolist()
content = tokenizer.decode(output_ids, skip_special_tokens=True)
display_conversation(messages, ds, content)


### Conversation

**System:** You are given the dialogue state for the current turn.
The dialogue state contains information about the user's intent and the extracted slot-value pairs:
{
    "intent": "...", 
    "slots": {
        "slot": "value"
    }
}

Based on the provided dialogue state, select the next best action from the list below:
- request_info(slot), if a slot value is missing (i.e., null)
- provide_info(intent, slot), provide the list of possible values for the requested slot for the given intent
- confirmation(intent), if all slots have been filled

Substitute 'slot' and 'intent' with the actual slot name and intent name from the dialogue state.
Only respond with the action in the exact format specified above. Do not include any additional text or explanation.


**User:** {
  "intent": "pizza_ordering",
  "slots": {
    "pizza_size": "medium",
    "pizza_type": "margherita",
    "pizza_count": "1"
  }
}

**Assistant:** confirmation(pizza_ordering)

### Exercise

Test whether the following prompt also works for:
- `drink_ordering`
- `out_of_domain`

#### Questions
How can you handle `out_of_domain`? 

#### (Possible) Solutions
1. Extend the prompt
2. Create an ad hoc prompt
3. Create a deterministic option for this case (not always a solution!)

### Exercise

Extend the prompt to
- call external components (e.g., API, or databases)
- recover from errors (e.g., when slots are not valid or a component is not reachable)

(Optional) Code a small function to verify whether the values from the NLU are valid

## Evaluation

Question: *How can we evalute the DM component?*

Think about:
1. What is the input
2. What is the expected output

### Example
 
Given the current DS:
```json
{
    "intent": "pizza_ordering", 
    "slots": {
        "pizza_type": "margherita",
        "pizza_size": null,
        "pizza_count": null
    }
}
```

what would you do next?

#### Solution
The DM should select the best action based on the current DS.

We can check whether the predicted action matches the ground-truth.

In [7]:
def check_actions(pred, gt):
    return pred.strip().lower() == gt.strip().lower()

# Example usage
predicted_action = "confirmation(pizza_ordering)"
ground_truth_action = "confirmation(pizza_ordering)"
print(check_actions(predicted_action, ground_truth_action))

True
